# Preprocesamiento y Modelado con PySpark
### Lending Club Loan Data

**Contenido:** Preprocesamiento completo + RandomForestClassifier con búsqueda manual de hiperparámetros.

> **Prerequisito:** Tener `df_clean.csv` generado desde el notebook de EDA.

---

## 1. Librerías e Inicialización

In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder,
                                 VectorAssembler, StandardScaler)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation    import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

sns.set_theme(style='whitegrid')

spark = (SparkSession.builder
         .appName('LendingClub_PySpark')
         .config('spark.driver.memory', '6g')
         .config('spark.driver.maxResultSize', '2g')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.executor.heartbeatInterval', '60s')
         .config('spark.network.timeout', '300s')
         .getOrCreate())

spark.sparkContext.setLogLevel('WARN')
print(f'✅ SparkSession iniciada — Spark {spark.version}')

## 2. Carga del Dataset

In [ ]:
sdf_raw = (spark.read
           .option('header', 'true')
           .option('inferSchema', 'false')
           .csv('df_clean.csv'))

print(f'Columnas: {len(sdf_raw.columns)}')
sdf_raw.show(3)

---
# Parte 1: Preprocesamiento

## 3. Selección de Variables

In [ ]:
CANDIDATE_FEATURES = [
    'loan_amnt', 'int_rate', 'fico_range_high', 'emp_length',
    'annual_inc', 'purpose', 'home_ownership', 'dti',
    'term', 'sub_grade', 'verification_status',
    'open_acc', 'pub_rec', 'revol_util', 'mort_acc',
    'installment', 'default'
]

available = [c for c in CANDIDATE_FEATURES if c in sdf_raw.columns]
sdf = sdf_raw.select(available)

# Castear numéricas con try_cast para tolerar valores malformados
NUMERIC_CANDIDATES = [
    'loan_amnt', 'int_rate', 'fico_range_high', 'emp_length',
    'annual_inc', 'dti', 'open_acc', 'pub_rec',
    'revol_util', 'mort_acc', 'installment'
]
for col in NUMERIC_CANDIDATES:
    if col in sdf.columns:
        sdf = sdf.withColumn(col, F.expr(f"try_cast({col} as double)"))

# Castear target: '0.0' string → float → int
sdf = sdf.withColumn("default", F.expr("try_cast(default as float)").cast("int"))
sdf = sdf.dropna()
sdf = sdf.filter(F.col("default").isin([0, 1]))

# Identificar tipos
CAT_COLS = [f.name for f in sdf.schema.fields
            if isinstance(f.dataType, StringType) and f.name != 'default']
NUM_COLS = [f.name for f in sdf.schema.fields
            if not isinstance(f.dataType, StringType) and f.name != 'default']

# Limitar a 50,000 filas para evitar OOM en RandomForest
sdf = sdf.limit(50000)
sdf = sdf.repartition(4)
sdf.cache()

print(f'Registros disponibles: {sdf.count():,}')
print(f'Numéricas  ({len(NUM_COLS)}): {NUM_COLS}')
print(f'Categóricas({len(CAT_COLS)}): {CAT_COLS}')
sdf.show(3)

## 4. División Train/Test — Estratificada

PySpark no tiene `stratify` nativo. Se divide cada clase por separado y luego se unen los conjuntos.

In [ ]:
sdf_0 = sdf.filter(F.col('default') == 0)
sdf_1 = sdf.filter(F.col('default') == 1)

train_0, test_0 = sdf_0.randomSplit([0.8, 0.2], seed=42)
train_1, test_1 = sdf_1.randomSplit([0.8, 0.2], seed=42)

train_sdf = train_0.union(train_1)
test_sdf  = test_0.union(test_1)

n_train = train_sdf.count()
n_test  = test_sdf.count()

print(f'Train: {n_train:,} filas')
print(f'Test : {n_test:,} filas')
print()
print('Distribución de clases — Train:')
train_sdf.groupBy('default').count().orderBy('default').show()
print('Distribución de clases — Test:')
test_sdf.groupBy('default').count().orderBy('default').show()

## 5. Pipeline de Preprocesamiento

### 5.1 StringIndexer — categóricas a índice numérico

In [ ]:
indexers = [
    StringIndexer(inputCol=col, outputCol=col + '_idx', handleInvalid='keep')
    for col in CAT_COLS
]
print(f'StringIndexers: {[col + "_idx" for col in CAT_COLS]}')

### 5.2 OneHotEncoder — vectores binarios

In [ ]:
encoders = [
    OneHotEncoder(inputCol=col + '_idx', outputCol=col + '_ohe')
    for col in CAT_COLS
]
print(f'OneHotEncoders: {[col + "_ohe" for col in CAT_COLS]}')

### 5.3 VectorAssembler + StandardScaler

In [ ]:
assembler_inputs = NUM_COLS + [col + '_ohe' for col in CAT_COLS]

assembler = VectorAssembler(
    inputCols  = assembler_inputs,
    outputCol  = 'features_raw',
    handleInvalid = 'skip'
)

scaler = StandardScaler(
    inputCol  = 'features_raw',
    outputCol = 'features',
    withMean  = True,
    withStd   = True
)

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

print('Pipeline de preprocesamiento:')
for i, stage in enumerate(prep_pipeline.getStages()):
    print(f'  Stage {i+1}: {type(stage).__name__}')

### 5.4 Ajustar y transformar

In [ ]:
print('Ajustando pipeline de preprocesamiento...')
t0 = time.time()

prep_model = prep_pipeline.fit(train_sdf)
train_prep = prep_model.transform(train_sdf)
test_prep  = prep_model.transform(test_sdf)

print(f'✅ Pipeline ajustado en {time.time()-t0:.2f} s')
print()
train_prep.select('default', 'features').show(3, truncate=70)

---
# Parte 2: Modelado con PySpark

## 6. Búsqueda Manual de Hiperparámetros

Se prueban las mismas combinaciones que en scikit-learn para garantizar comparabilidad.

In [ ]:
# maxDepth=15 causa OutOfMemoryError en la JVM con datasets grandes
# Se limita a maxDepth=10 y se agregan parámetros para reducir uso de memoria
n_estimators_list = [10, 50, 100]
max_depth_list    = [5, 10]

combinations = list(itertools.product(n_estimators_list, max_depth_list))

evaluator_auc = BinaryClassificationEvaluator(
    labelCol         = 'default',
    rawPredictionCol = 'rawPrediction',
    metricName       = 'areaUnderROC'
)

results = []

print(f'Combinaciones a probar: {len(combinations)}')
print(f'{"n_est":>6} {"max_d":>6} {"ROC_AUC":>10} {"Tiempo(s)":>10}')
print('-' * 38)

for n_est, max_d in combinations:
    rf = RandomForestClassifier(
        labelCol        = 'default',
        featuresCol     = 'features',
        numTrees        = n_est,
        maxDepth        = max_d,
        maxBins         = 32,
        subsamplingRate = 0.8,
        seed            = 42
    )
    t_start = time.time()
    model   = rf.fit(train_prep)
    t_train = time.time() - t_start

    preds = model.transform(test_prep)
    auc   = evaluator_auc.evaluate(preds)

    results.append({
        'n_estimators' : n_est,
        'max_depth'    : max_d,
        'roc_auc'      : round(auc, 4),
        'train_time_s' : round(t_train, 2),
        'model'        : model,
        'predictions'  : preds
    })
    print(f'{n_est:>6} {max_d:>6} {auc:>10.4f} {t_train:>10.2f}')

### 6.1 Heatmap de resultados

In [ ]:
df_results = pd.DataFrame([{k: v for k, v in r.items()
                              if k not in ('model', 'predictions')}
                             for r in results])

pivot = df_results.pivot(index='max_depth', columns='n_estimators', values='roc_auc')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=0.5, ax=ax, annot_kws={'size': 11})
ax.set_title('Búsqueda manual — ROC AUC por hiperparámetros (PySpark)', fontweight='bold')
ax.set_xlabel('n_estimators (numTrees)')
ax.set_ylabel('max_depth (maxDepth)')
plt.tight_layout()
plt.savefig('spark_01_gridsearch_heatmap.png', bbox_inches='tight')
plt.show()

print(pivot.to_string())

## 7. Selección del Mejor Modelo

In [ ]:
best_result = max(results, key=lambda r: r['roc_auc'])
best_model  = best_result['model']
best_preds  = best_result['predictions']

print(f'Mejores hiperparámetros:')
print(f'  n_estimators (numTrees): {best_result["n_estimators"]}')
print(f'  max_depth               : {best_result["max_depth"]}')
print(f'  ROC AUC (test)          : {best_result["roc_auc"]}')
print(f'  Tiempo entrenamiento    : {best_result["train_time_s"]} s')

## 8. Predicción y Tiempo

In [ ]:
best_rf = RandomForestClassifier(
    labelCol        = 'default',
    featuresCol     = 'features',
    numTrees        = best_result['n_estimators'],
    maxDepth        = best_result['max_depth'],
    maxBins         = 32,
    subsamplingRate = 0.8,
    seed            = 42
)

t_train_start = time.time()
final_model   = best_rf.fit(train_prep)
t_train_final = time.time() - t_train_start

t_pred_start = time.time()
final_preds  = final_model.transform(test_prep)
final_preds.count()
t_pred_final = time.time() - t_pred_start

print(f'⏱️  Tiempo entrenamiento: {t_train_final:.2f} s')
print(f'⏱️  Tiempo predicción   : {t_pred_final:.4f} s')

## 9. Métricas de Evaluación

In [ ]:
# ROC AUC
auc = evaluator_auc.evaluate(final_preds)

# Accuracy, Precision, Recall, F1
mc_eval = MulticlassClassificationEvaluator(labelCol='default', predictionCol='prediction')

acc  = mc_eval.evaluate(final_preds, {mc_eval.metricName: 'accuracy'})
prec = mc_eval.evaluate(final_preds, {mc_eval.metricName: 'weightedPrecision'})
rec  = mc_eval.evaluate(final_preds, {mc_eval.metricName: 'weightedRecall'})
f1   = mc_eval.evaluate(final_preds, {mc_eval.metricName: 'f1'})

print('=' * 42)
print('    MÉTRICAS — RandomForest (PySpark)')
print('=' * 42)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-score  : {f1:.4f}')
print(f'  ROC AUC   : {auc:.4f}')
print('-' * 42)
print(f'  Tiempo entrenamiento: {t_train_final:.2f} s')
print(f'  Tiempo predicción   : {t_pred_final:.4f} s')
print('=' * 42)

## 10. Matriz de Confusión

In [ ]:
# Convertir a Pandas para visualizar
preds_pd = final_preds.select('default', 'prediction').toPandas()

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve

cm = confusion_matrix(preds_pd['default'], preds_pd['prediction'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Fully Paid', 'Charged Off'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión', fontweight='bold')

# Curva ROC — usar probabilidad de clase positiva
proba_pd = final_preds.select('default', 'probability').toPandas()
proba_pd['prob_1'] = proba_pd['probability'].apply(lambda v: float(v[1]))

fpr, tpr, _ = roc_curve(proba_pd['default'], proba_pd['prob_1'])
axes[1].plot(fpr, tpr, color='#8e44ad', lw=2, label=f'ROC AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#8e44ad')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC', fontweight='bold')
axes[1].legend()

plt.suptitle('Evaluación — RandomForest PySpark', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('spark_02_evaluation.png', bbox_inches='tight')
plt.show()

## 11. Importancia de Variables

In [ ]:
feature_names = NUM_COLS + [col + '_ohe' for col in CAT_COLS]
importances   = final_model.featureImportances.toArray()

# El vector de features puede tener más dimensiones que feature_names (por OHE)
# Usar solo las primeras len(feature_names) por simplicidad
n = min(len(feature_names), len(importances))
imp_series = pd.Series(importances[:n], index=feature_names[:n]).sort_values(ascending=False)

top15 = imp_series.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('RdYlGn_r', len(top15))
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Top 15 Variables más Importantes — RandomForest PySpark', fontweight='bold')
plt.tight_layout()
plt.savefig('spark_03_feature_importance.png', bbox_inches='tight')
plt.show()

## 12. Resumen Final

In [ ]:
metrics_spark = {
    'Modelo'        : 'RandomForest (PySpark)',
    'n_estimators'  : best_result['n_estimators'],
    'max_depth'     : best_result['max_depth'],
    'Accuracy'      : round(acc,  4),
    'Precision'     : round(prec, 4),
    'Recall'        : round(rec,  4),
    'F1-score'      : round(f1,   4),
    'ROC AUC'       : round(auc,  4),
    'Tiempo_train_s': round(t_train_final, 2),
    'Tiempo_pred_s' : round(t_pred_final,  4)
}

pd.DataFrame([metrics_spark])

In [ ]:
import json
with open('metrics_spark.json', 'w') as f:
    json.dump(metrics_spark, f, indent=2)

print('✅ Métricas guardadas en metrics_spark.json')
spark.stop()
print('✅ SparkSession cerrada')